# Qwen3-VL-4B: effect of the fovea's (R, a) on accuracy

A slim copy of `colab_vlm_egoeverything_qwen3.ipynb` for one job: **try different (R, a) ring settings**
and compare their accuracy. Setup, gate, send policy and frames are the same code as that notebook.

- **R** = the fovea's radius (degrees), **a** = how fast resolution falls off toward the periphery
  (`q3.foveated_cells(gu, gv, a=a, R_deg=R)`). The token count per frame follows from (R, a) and the gaze.
- Each pick is scored as **fovea at gaze**; optionally also as **Qwen3-VL's own input downscaled to the
  same tokens** (the fair control) and as **fovea at the image centre**.
- Results go to their own file on Drive (`*_RaSweep.jsonl`), keyed by (R, a), so the main notebook's
  results are untouched and a re-run only scores the new picks.

**Edit the settings cell, then Runtime -> Run all.** GPU: A100 or L4 (Qwen3-VL-4B in bf16).

## 0. Settings: edit here

**R** = how far from the gaze (degrees) the ring pattern reaches: both ring boundaries lie between the
gaze and R, so a larger R pushes them out and keeps more of the frame sharp (more tokens).
**a** = how the rings are spaced: a small a packs them near the gaze (small sharp fovea, fast drop to
coarse), a large a spreads them evenly (bigger sharp area).
Inside ring 1: full-resolution 32 px cells; ring 1 to ring 2: 64 px; beyond ring 2: 128 px.

The default grid: approximate tokens per frame (ring 1 / ring 2 radius in degrees); the exact
counts are printed in section 7.

| R \ a | 40 | 160 | 320 |
|---|---|---|---|
| **70** | 42 (16°/39°) | 46 (21°/44°) | 52 (22°/45°) |
| **180** | 99 (31°/85°) | **147 (46°/104°), centre** | 166 (51°/111°) |
| **240** | 117 (37°/106°) | 198 (57°/135°) | 216 (66°/145°) |

The old four picks, (70, 160), (180, 40), (180, 160) and (240, 160), are part of this grid.
9 fovea + 9 same-token uniform + stock = 19 conditions on 204 questions (DEMO).

In [ ]:
# (R, a) picks to test; add, remove or change freely
# 3 x 3 grid centred on (R=180, a=160): every direction from the centre (both up, both down, mixed)
PICKS = [dict(R=R, a=a) for R in (70, 180, 240) for a in (40, 160, 320)]

MATCHED_UNIFORM = True    # also score Qwen3-VL's own input downscaled to each pick's token count
CENTRE_CONTROL  = False   # also score each pick with the fovea at the image centre instead of the gaze
DEMO   = True             # True: the 4 test Aria videos (204 questions); False: all 12 gate videos (834)
POLICY = 'D+P'            # send policy for the frames: 'D', 'D+P' or 'D+P all'

## 1. Clone, mount, install, model check

In [ ]:
import os, sys, json, subprocess

REPO, BRANCH = '/content/GazeVLM', 'eyehead-gate'     # 'main' once merged
URL = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
assert os.path.exists('vlmeval/qwen3_fovea.py'), f'vlmeval/qwen3_fovea.py is not on branch {BRANCH!r}'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pytest'], check=True)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/GazeVLM'
FOLDS     = os.path.join(DRIVE_DIR, 'folds')
RUNS      = os.path.join(DRIVE_DIR, 'runs')
VLM_DIR   = os.path.join(DRIVE_DIR, 'vlm', 'egoeverything')
FRAMES_TAR = os.path.join(VLM_DIR, 'frames_tar')          # one .tar per video and size, reused later
LOCAL_HI = '/content/frames_1344'
os.makedirs(VLM_DIR, exist_ok=True)

CKPT_FULL = os.path.join(RUNS, 'eyehead')                  # the Full model's 5 fold checkpoints
missing = [i for i in range(1, 6) if not os.path.exists(os.path.join(CKPT_FULL, f'fold{i}.pt'))]
assert not missing, f'missing fold checkpoints {missing} in {CKPT_FULL} -- run the eyehead notebook first'
print('Full model checkpoints: 5 folds found in', CKPT_FULL)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
# Qwen3-VL-4B (transformers-native); a Hugging Face token is optional (rate limits)
from huggingface_hub import login, model_info
from vlmeval import qwen3_fovea as q3
try:
    from google.colab import userdata
    _tok = userdata.get('HF_TOKEN')
    if _tok:
        login(token=_tok)
except Exception:
    pass
MODEL_ID = None
for cand in q3.MODEL_CANDIDATES:
    try:
        model_info(cand)
        MODEL_ID = cand
        break
    except Exception as ex:
        print(f'   {cand}: not available ({type(ex).__name__})')
assert MODEL_ID, f'none of {q3.MODEL_CANDIDATES} is reachable'
import transformers
print('model:', MODEL_ID, '| transformers', transformers.__version__)
assert hasattr(transformers, 'Qwen3VLForConditionalGeneration'), \
    'this transformers has no Qwen3-VL (needs >= 4.57): pip install -U transformers, then restart the runtime'

## 2. The AEA download-links JSON (refreshes the Drive copy if expired)

In [ ]:
from vlmeval.frames import links_fresh, install_links

FORCE_UPLOAD = False
LINKS = os.path.join(DRIVE_DIR, 'aea_download_urls.json')
OLD_COPIES = [LINKS] + [p for p in ['/content/drive/MyDrive/aea/aea_download_urls.json']
                        if os.path.exists(p)]

ok, msg = links_fresh(LINKS) if os.path.exists(LINKS) else (False, 'not on Drive')
print('Drive copy:', msg)
if FORCE_UPLOAD or not ok:
    from google.colab import files
    print('upload the fresh aea_download_urls.json ...')
    up = files.upload()
    name = list(up.keys())[0]
    ok, msg = links_fresh(name)
    print('uploaded file:', msg)
    assert ok, 'the uploaded JSON does not work either -- download a fresh one from projectaria.com/datasets/aea'
    install_links(name, OLD_COPIES)
    os.remove(name)                      # the copies live on Drive; keep the repo clean
    for p in OLD_COPIES:
        print('   replaced', p)
links = json.load(open(LINKS))
print(f"using {LINKS}: {len(links['sequences'])} sequences")
print('(the eyehead notebook\'s IMU calibration, section 3, can now download its VRS too)')

## 3. EgoEverything questions on the gate's videos

In [ ]:
from vlmeval import egoev

qs_all = egoev.load_questions()
aea = qs_all[qs_all.video.str.startswith('loc')]
clips_all, test = egoev.gate_clips(FOLDS, test_only=False)
gate_videos = {c['sequence'] for c in clips_all}
test_videos = {c['sequence'] for c in clips_all if c['is_test']}

# DEMO = True: only the 4 locked TEST videos (204 questions), to check that everything works.
# DEMO = False: all 12 gate videos with EgoEverything questions (834 questions).
DEMO = DEMO          # set in the settings cell above
qs = aea[aea.video.isin(gate_videos)].reset_index(drop=True)
if DEMO:
    qs = qs[qs.video.isin(test_videos)].reset_index(drop=True)
clips = [c for c in clips_all if c['sequence'] in set(qs.video)]
videos = [c['sequence'] for c in clips]
n_frames = {c['sequence']: len(c['frame_no']) for c in clips}
print(f'EgoEverything: {len(qs_all):,} questions, {len(aea):,} on AEA ({aea.video.nunique()} videos)')
print(f'on the gate videos: {len(qs):,} questions on {len(videos)} videos '
      f'({qs.video.isin(test_videos).sum()} questions on {len(set(videos) & test_videos)} test videos)\n')

per = qs.groupby('video').agg(questions=('qid', 'size'), t0_max=('t0', 'max'),
                              median_dist2gaze=('dist2gaze', 'median'))
per['video_s'] = [n_frames[v] for v in per.index]
per['split'] = ['test' if v in test_videos else 'train (held out by its fold)' for v in per.index]
display(per.sort_values('questions', ascending=False).round(1))
print('answer letters:', qs.answer.value_counts().sort_index().to_dict())

## 4. Full gate + send policy: which frames are sent

In [ ]:
from eyehead.oof import oof_predictions

OOF = os.path.join(VLM_DIR, 'oof_preds_full.csv')
if os.path.exists(OOF):
    oof = pd.read_csv(OOF)
    print('loaded', OOF)
else:
    oof = oof_predictions(FOLDS, CKPT_FULL)
    oof.to_csv(OOF, index=False)
    print('wrote', OOF)

o12 = oof[oof.sequence.isin(videos)]
print(f"held-out 4-class accuracy on these {len(videos)} videos: {(o12.pred == o12.quad).mean():.3f}")

labels = {'full': egoev.video_labels(clips, oof)}
# Temporal policy: 'D' = send when the view settles + on each REFIXATION;
# 'D+P' = D + the first frame of every PURSUIT run; 'D+P all' = D + every PURSUIT frame.
POLICY = POLICY      # set in the settings cell above
POLICY_TAG = {'D': 'D', 'D+P': 'DP', 'D+P all': 'DPall'}[POLICY]   # separate result files per policy
sel = {'D: full': egoev.select_conditions(clips, labels, main='full')[f'{POLICY}: full']}
D = sel['D: full']
per = pd.DataFrame({'video_s': n_frames, 'frames sent': {v: len(D[v]) for v in videos}})
per['one frame every (s)'] = (per.video_s / per['frames sent']).round(1)
display(per)
print(f"policy {POLICY} sends {60 * per['frames sent'].sum() / per.video_s.sum():.1f} frames/min "
      f"({per['frames sent'].mean():.1f} per video)")

## 5. Frames and gaze

In [ ]:
from vlmeval.frames import ensure_frames, frame_path
from vlmeval import qwen_fovea as qf, spatial as sp

metas = {}
for v in videos:
    metas[v] = ensure_frames(v, links, LOCAL_HI, FRAMES_TAR, size=qf.IMG)
    m = metas[v]
    flag = 'OK' if m['n'] >= n_frames[v] else f'!! only {m["n"]} frames, labels need {n_frames[v]}'
    print(f"   {v:<26} {m['n']:4d} frames  {flag}")

def frame_time(v, k):
    return metas[v]['ts'][k] if k < metas[v]['n'] else k

tracks, n_drop = sp.gaze_tracks(FOLDS, videos, frame_time)
frames_of = {}
for v in videos:
    m = metas[v]
    frames_of[v] = []
    for f in D[v]:
        if f >= m['n']:
            continue
        t = float(m['ts'][f])
        gz_ = sp.gaze_fixation(tracks[v], t) if v in tracks else None
        frames_of[v].append(dict(key=(v, int(f)), path=frame_path(m, f), t=t,
                                 gaze=tuple(map(float, gz_)) if gz_ is not None else None))
items = [dict(qid=int(q.qid), video=q.video, question=q.question, options=list(q.options),
              answer=q.answer, frames=frames_of[q.video])
         for q in qs.sort_values(['video', 'qid']).itertuples()]
nf = np.array([len(it['frames']) for it in items])
print(f"{len(items)} questions on {len(videos)} videos; frames per question: mean {nf.mean():.1f}, max {nf.max()}; "
      f"gaze missing on {sum(fr['gaze'] is None for v in videos for fr in frames_of[v])} of "
      f"{sum(len(frames_of[v]) for v in videos)} sent frames")

## 6. Load Qwen3-VL-4B and check parity with its own pipeline

If parity fails, the foveated path does not match the stock model and nothing below is trustworthy.

In [ ]:
from vlmeval import qwen_mcq                      # same prompt (frame times, question, options)
model, proc, letter_ids = q3.load_model(MODEL_ID)
print(MODEL_ID, f'{torch.cuda.memory_allocated()/1e9:.1f} GB on GPU')

q0 = qs.iloc[0]
m0 = metas[q0.video]
k0 = int(np.argmin(np.abs(np.array(m0['ts']) - q0.t0)))
img0 = qf.load_frame01(frame_path(m0, k0))
rep = q3.parity_report(model, proc, letter_ids, img0, q0.question, q0.options, qwen_mcq.build_messages)
ok = (rep['pixel_max_abs_diff'] < 2e-2 and rep['vision_fp32_cos_min'] > 0.9999
      and rep['deepstack_fp32_cos_min'] > 0.9999 and rep['tokens_ids_equal']
      and rep['llm_probs_max_abs_diff'] < 1e-3 and rep['llm_same_answer'])
for k_, v_ in rep.items():
    print(f'   {k_:<30} {v_}')
assert ok, 'parity failed: do not trust the foveated results'
print('parity OK')

## 7. The picks: token budgets and layouts

Tokens per frame are the mean over the sent frames (they depend on where the gaze falls).
Qwen3-VL stock = 256 tokens per 512 px frame.

In [ ]:
PICKS = [dict(R=p['R'], a=p['a']) for p in PICKS]
assert len({(p['R'], p['a']) for p in PICKS}) == len(PICKS), 'duplicate (R, a) in PICKS'
gz_D = [fr['gaze'] or (0.5, 0.5) for v in videos for fr in frames_of[v]]
for p in PICKS:
    p['tokens'] = float(np.mean([len(q3.foveated_cells(*g_, a=p['a'], R_deg=p['R'])) for g_ in gz_D]))
budget = pd.DataFrame([{'R': p['R'], 'a': p['a'], 'tokens / frame': p['tokens'],
                        'saving vs 256 (%)': 100 * (1 - p['tokens'] / 256)} for p in PICKS])
display(budget.style.format({'tokens / frame': '{:.1f}', 'saving vs 256 (%)': '{:.1f}'}).hide(axis='index'))

COLS = {32: 'white', 64: 'yellow', 128: 'deepskyblue'}
v_ = videos[0]
fr_ = frames_of[v_][len(frames_of[v_]) // 2]
im_ = qf.resize01(qf.load_frame01(fr_['path']), q3.IMG).permute(1, 2, 0).numpy()
g_ = fr_['gaze'] or (0.5, 0.5)
n = len(PICKS) + 1
fig, axs = plt.subplots(1, n, figsize=(3.4 * n, 3.7))
axs[0].imshow(im_); axs[0].set_title('stock: 256 tokens', fontsize=8); axs[0].axis('off')
for a_, p in zip(axs[1:], PICKS):
    cells = q3.foveated_cells(*g_, a=p['a'], R_deg=p['R'])
    a_.imshow(im_)
    for y, x, s in cells:
        a_.add_patch(mpatches.Rectangle((x, y), s, s, fill=False, ec=COLS.get(s, 'red'), lw=0.6))
    a_.plot(g_[0] * q3.IMG, g_[1] * q3.IMG, 'r+', ms=12, mew=2)
    a_.set_title(f"R={p['R']}, a={p['a']}: {len(cells)} tokens", fontsize=8); a_.axis('off')
plt.suptitle('one token per box: white 32 px (Qwen3-VL own token), yellow 64 px, blue 128 px; + = gaze', fontsize=9)
plt.tight_layout(); plt.show()

## 8. Score

Answer = argmax of the raw next-token logits over A-E. Resumable: conditions already in the
results file are skipped, so adding a pick only scores that pick.

In [ ]:
fov = lambda p, src='gaze': f"Qwen3 fovea {src} (R={p['R']}, a={p['a']})"
UNI = lambda p: f"Qwen3 uniform downscaled, same tokens as (R={p['R']}, a={p['a']})"
STOCK = 'Qwen3 stock 512 (256 tokens)'

CONDS = {STOCK: ('uniform', q3.IMG)}
for p in PICKS:
    CONDS[fov(p)] = ('fovea', dict(a=p['a'], R=p['R'], gaze='gaze'))
    if MATCHED_UNIFORM:
        CONDS[UNI(p)] = ('uniform_match', fov(p))
    if CENTRE_CONTROL:
        CONDS[fov(p, 'centre')] = ('fovea', dict(a=p['a'], R=p['R'], gaze='centre'))
print(f'{len(items)} questions x {len(CONDS)} conditions:', *CONDS, sep='\n   ')

OUT = os.path.join(VLM_DIR, f"{MODEL_ID.split('/')[-1]}_{POLICY_TAG}_{'demo' if DEMO else 'all'}_RaSweep.jsonl")
q3.run_questions(model, proc, letter_ids, items, CONDS, OUT, qwen_mcq.build_messages)

## 9. Results

One row per (R, a). Accuracy with a 95% interval from resampling videos; differences are paired on
the same questions (a positive value = the fovea is better). Green = higher accuracy.

In [ ]:
res = egoev.load_results(OUT)
res = res[res.video.isin(videos) & res.condition.isin(CONDS)]
bad = res[res['error'].notna()] if 'error' in res else res.iloc[:0]
if len(bad):
    print(f'{len(bad)} rows not scored, left out:'); display(bad.groupby(['condition', 'error']).size())
res = res[res['error'].isna()].copy()
res['correct'] = res['correct'].astype(int)
acc = egoev.accuracy_table(res)
tok = res.groupby('condition').n_image_tokens.mean() / acc['frames_per_q']
have = set(res.condition)

def pair(a_, b_):
    if not {a_, b_} <= have:
        return np.nan, ''
    t = egoev.paired_test(res, a_, b_)
    return t['diff'], f"[{t['ci_low']:+.1f}, {t['ci_high']:+.1f}]"

rows = []
for p in PICKS:
    f = fov(p)
    if f not in acc.index:
        continue
    r = {'R': p['R'], 'a': p['a'], 'tokens / frame': tok[f], 'saving vs 256 (%)': 100 * (1 - tok[f] / 256),
         'accuracy (%)': acc.accuracy[f], '95% CI': f"[{acc.ci_low[f]:.1f}, {acc.ci_high[f]:.1f}]"}
    r['vs stock'], r['vs stock CI'] = pair(f, STOCK)
    if MATCHED_UNIFORM:
        r['uniform same tokens (%)'] = acc.accuracy.get(UNI(p), np.nan)
        r['vs uniform'], r['vs uniform CI'] = pair(f, UNI(p))
    if CENTRE_CONTROL:
        r['centre fovea (%)'] = acc.accuracy.get(fov(p, 'centre'), np.nan)
        r['gaze vs centre'], r['gaze vs centre CI'] = pair(f, fov(p, 'centre'))
    rows.append(r)
table = pd.DataFrame(rows).sort_values('tokens / frame').reset_index(drop=True)

print(f"Qwen3-VL-4B, {res.qid.nunique()} questions on {res.video.nunique()} videos, policy {POLICY} frames")
if STOCK in acc.index:
    print(f"stock 512 px (256 tokens / frame): {acc.accuracy[STOCK]:.1f}%  "
          f"[{acc.ci_low[STOCK]:.1f}, {acc.ci_high[STOCK]:.1f}]")
num = [c for c in table.columns if table[c].dtype.kind == 'f']
fmt = {c: '{:.1f}' for c in num}
fmt.update({c: '{:+.1f}' for c in ('vs stock', 'vs uniform', 'gaze vs centre') if c in table})
acc_cols = [c for c in ('accuracy (%)', 'uniform same tokens (%)', 'centre fovea (%)') if c in table]
diff_cols = [c for c in ('vs stock', 'vs uniform', 'gaze vs centre') if c in table]
sty = (table.style.format(fmt, na_rep='-').hide(axis='index')
       .background_gradient(cmap='RdYlGn', subset=acc_cols, axis=None)
       .background_gradient(cmap='RdYlGn', subset=diff_cols, axis=None, vmin=-5, vmax=5)
       .set_properties(**{'text-align': 'center'})
       .set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]}])
       .set_caption('Fovea at gaze, per (R, a); diffs in accuracy points, fovea minus the other'))
display(sty)
table.to_csv(OUT.replace('.jsonl', '_table.csv'), index=False)

fig, ax = plt.subplots(figsize=(7, 4.4))
for p in PICKS:
    f = fov(p)
    if f in acc.index:
        ax.errorbar(tok[f], acc.accuracy[f], yerr=[[acc.accuracy[f] - acc.ci_low[f]], [acc.ci_high[f] - acc.accuracy[f]]],
                    fmt='o', capsize=3, color='C0')
        ax.annotate(f"R={p['R']}, a={p['a']}", (tok[f], acc.accuracy[f]), textcoords='offset points',
                    xytext=(5, 5), fontsize=7)
    if MATCHED_UNIFORM and UNI(p) in acc.index:
        ax.plot(tok[UNI(p)], acc.accuracy[UNI(p)], 's', color='C1', mfc='none')
if STOCK in acc.index:
    ax.axhline(acc.accuracy[STOCK], color='k', ls=':', lw=1)
ax.plot([], [], 'o', color='C0', label='fovea at gaze (95% CI over videos)')
if MATCHED_UNIFORM:
    ax.plot([], [], 's', color='C1', mfc='none', label='uniform downscaled, same tokens')
ax.plot([], [], 'k:', label='stock 512 px (256 tokens)')
ax.set(xlabel='visual tokens per frame', ylabel='accuracy (%)', title=f'Qwen3-VL-4B, accuracy vs (R, a)')
ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

if table.R.nunique() > 1 and table.a.nunique() > 1:
    print('accuracy (%) by R (rows) and a (columns):')
    display(table.pivot_table(index='R', columns='a', values='accuracy (%)').style
            .format('{:.1f}', na_rep='-').background_gradient(cmap='RdYlGn', axis=None))

## Reading it

- **vs uniform** is the fair comparison: same tokens, only *where* the resolution goes differs.
- **vs stock** shows how much accuracy a pick gives up for its token saving.
- 204 questions (DEMO) give wide intervals (about +/- 3-4 points): treat differences under that as ties,
  and confirm a chosen (R, a) with `DEMO = False` (834 questions).